# Online Shoppers Purchasing Intention Prediction

## Machine Learning Assignment 2

### Problem Statement

The objective of this project is to develop machine learning classification
models that can predict whether an online shopping session will result in a
purchase.

The Online Shoppers Purchasing Intention dataset is used for this study.
The target variable is Revenue, where True represents a successful purchase
and False represents a session without a purchase.

The following classification algorithms are evaluated:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors
4. Naive Bayes
5. Random Forest

The models are compared using Accuracy, AUC, Precision, Recall, F1 Score
and Matthews Correlation Coefficient (MCC).

In [34]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("ML_Assignment_2/online_shoppers_intention.csv")

# Basic dataset information
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Revenue"].value_counts())

print("\nTarget Distribution Percentage:")
print((df["Revenue"].value_counts(normalize=True) * 100).round(2))

df.head()

Dataset Shape: (12330, 18)

Column Names:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

Missing Values:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

Target Distribution:
Revenue
False    10422
True      1908
Name: count, dtype: int64

Target Distribution Percent

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [35]:
# Separate features and target
X = df.drop("Revenue", axis=1)
y = df["Revenue"].astype(int)

# Convert categorical features into numerical form
X = pd.get_dummies(
    X,
    columns=["Month", "VisitorType"],
    drop_first=True
)

# Convert Weekend Boolean value to integer
X["Weekend"] = X["Weekend"].astype(int)

print("Feature shape after preprocessing:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature shape after preprocessing: (12330, 26)
Target shape: (12330,)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,VisitorType_Other,VisitorType_Returning_Visitor
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,True,False,False,False,False,False,False,False,False,True
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,...,True,False,False,False,False,False,False,False,False,True
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,True,False,False,False,False,False,False,False,False,True
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,...,True,False,False,False,False,False,False,False,False,True
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,...,True,False,False,False,False,False,False,False,False,True


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)
print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Data Shape: (9864, 26)
Testing Data Shape: (2466, 26)
Training Target Shape: (9864,)
Testing Target Shape: (2466,)


In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Define required models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

results = []

for model_name, model in models.items():

    # Logistic Regression and KNN use scaled data
    if model_name in ["Logistic Regression", "KNN"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test_scaled)[:, 1]

    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append([
        model_name,
        accuracy,
        auc,
        precision,
        recall,
        f1,
        mcc
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "ML Model Name",
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
)

results_df.round(4)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8812,0.8873,0.7432,0.3560,0.4814,0.4603
1,Decision Tree,0.8646,0.7360,0.5645,0.5497,0.5570,0.4772
2,KNN,0.8686,0.7725,0.6368,0.3534,0.4545,0.4085
3,Naive Bayes,0.7960,0.8095,0.3951,0.5969,0.4755,0.3670
4,Random Forest,0.8978,0.9179,0.7321,0.5366,0.6193,0.5710


## Model Performance Observations

The performance of all classification models was compared using Accuracy,
AUC, Precision, Recall, F1 Score and Matthews Correlation Coefficient (MCC).

The dataset is imbalanced, with a much larger number of non-purchase
sessions than purchase sessions. Therefore, accuracy alone is not sufficient
to determine the best model. Metrics such as Recall, F1 Score, AUC and MCC
are also considered while comparing the models.

### Observations

**Logistic Regression:**  
Logistic Regression achieved good accuracy and AUC. It also showed high
precision, which means that when the model predicted a purchase, it was
often correct. However, its recall was relatively low, indicating that it
missed many actual purchase sessions.

**Decision Tree:**  
The Decision Tree achieved lower accuracy and AUC compared with Logistic
Regression and Random Forest. However, its recall was better than Logistic
Regression and KNN, showing that it identified more actual purchase cases.

**K-Nearest Neighbors:**  
KNN achieved reasonable accuracy but had relatively low recall and F1 Score.
This indicates that the model performed well for the majority class but was
less effective in identifying purchase sessions.

**Naive Bayes:**  
Naive Bayes produced the highest recall among the tested models. This means
it identified a relatively large number of actual purchases. However, its
precision and overall accuracy were lower, resulting in more false positive
predictions.

**Random Forest:**  
Random Forest produced the strongest overall performance. It achieved the
highest Accuracy, AUC, F1 Score and MCC among the tested models. It also
maintained a good balance between precision and recall.

**Overall Winner:**  
Random Forest is selected as the best performing model for this dataset
because it achieved the best overall balance across the evaluation metrics,
particularly Accuracy, AUC, F1 Score and MCC.

In [38]:
from sklearn.metrics import confusion_matrix, classification_report

for model_name, model in models.items():

    if model_name in ["Logistic Regression", "KNN"]:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))


Logistic Regression

Confusion Matrix:
[[2037   47]
 [ 246  136]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.98      0.93      2084
           1       0.74      0.36      0.48       382

    accuracy                           0.88      2466
   macro avg       0.82      0.67      0.71      2466
weighted avg       0.87      0.88      0.86      2466


Decision Tree

Confusion Matrix:
[[1922  162]
 [ 172  210]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92      2084
           1       0.56      0.55      0.56       382

    accuracy                           0.86      2466
   macro avg       0.74      0.74      0.74      2466
weighted avg       0.86      0.86      0.86      2466


KNN

Confusion Matrix:
[[2007   77]
 [ 247  135]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.96      0

In [39]:
# Create test data in the original dataset format
# This makes the CSV easier to use in the Streamlit application

_, raw_test_data = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Revenue"]
)

raw_test_data.to_csv("test_data.csv", index=False)

print("test_data.csv created successfully")
print("Test data shape:", raw_test_data.shape)

raw_test_data.head()

test_data.csv created successfully
Test data shape: (2466, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
4722,1,4.0,0,0.0,13,161.166667,0.024615,0.061538,0.000000,0.6,May,2,5,9,5,Returning_Visitor,False,False
6835,1,23.2,0,0.0,23,290.000000,0.000000,0.033333,0.000000,0.0,Jul,2,2,8,3,Returning_Visitor,False,False
5524,0,0.0,0,0.0,6,95.000000,0.028571,0.028571,86.336233,0.0,Oct,1,1,4,2,New_Visitor,False,True
663,0,0.0,0,0.0,2,17.000000,0.000000,0.100000,0.000000,0.0,Mar,1,1,8,3,Returning_Visitor,False,False
136,0,0.0,0,0.0,9,303.666667,0.005556,0.046296,0.000000,0.0,Feb,2,4,5,2,Returning_Visitor,False,False


In [40]:
import os
import joblib

os.makedirs("model", exist_ok=True)

joblib.dump(models["Logistic Regression"], "model/logistic_regression.pkl")
joblib.dump(models["Decision Tree"], "model/decision_tree.pkl")
joblib.dump(models["KNN"], "model/knn.pkl")
joblib.dump(models["Naive Bayes"], "model/naive_bayes.pkl")
joblib.dump(models["Random Forest"], "model/random_forest.pkl")

joblib.dump(scaler, "model/scaler.pkl")

print("All models and scaler saved successfully")

All models and scaler saved successfully
